In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType,ArrayType

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, explode_outer
from pyspark.sql.types import StructType, ArrayType

# ---------------------------
# Recursive function to flatten DataFrame
# ---------------------------
def flatten_df(nested_df):
    """
    Recursively flattens a nested DataFrame (structs and arrays).
    """
    flat_cols = []
    nested_cols = []

    for field in nested_df.schema.fields:
        field_name = field.name
        field_type = field.dataType

        if isinstance(field_type, StructType):
            # Expand struct fields
            for sub_field in field_type.fields:
                flat_cols.append(col(f"{field_name}.{sub_field.name}")
                                 .alias(f"{field_name}_{sub_field.name}"))
        elif isinstance(field_type, ArrayType) and isinstance(field_type.elementType, StructType):
            # Explode arrays of structs
            nested_df = nested_df.withColumn(field_name, explode_outer(col(field_name)))
            for sub_field in field_type.elementType.fields:
                flat_cols.append(col(f"{field_name}.{sub_field.name}")
                                 .alias(f"{field_name}_{sub_field.name}"))
        elif isinstance(field_type, ArrayType):
            # Explode arrays of primitive types
            nested_df = nested_df.withColumn(field_name, explode_outer(col(field_name)))
            flat_cols.append(col(field_name))
        else:
            flat_cols.append(col(field_name))

    flat_df = nested_df.select(flat_cols)

    # Check if more flattening is needed
    if any(isinstance(f.dataType, StructType) or isinstance(f.dataType, ArrayType)
           for f in flat_df.schema.fields):
        return flatten_df(flat_df)
    else:
        return flat_df




In [0]:
# ---------------------------
# Main execution
# ---------------------------
if __name__ == "__main__":
    spark = SparkSession.builder \
        .appName("FlattenNestedJSON") \
        .getOrCreate()

    # Example nested JSON data
    json_data = [
        {
            "id": 1,
            "name": {"first": "John", "last": "Doe"},
            "contacts": [
                {"type": "email", "value": "john@example.com"},
                {"type": "phone", "value": "1234567890"}
            ],
            "address": {"city": "New York", "zip": "10001"}
        },
        {
            "id": 2,
            "name": {"first": "Jane", "last": "Smith"},
            "contacts": [],
            "address": {"city": "Los Angeles", "zip": "90001"}
        }
    ]

    # Create DataFrame from JSON
    df = spark.createDataFrame(json_data)

    print("Original Nested Schema:")
    df.printSchema()

    # Flatten the DataFrame
    flat_df = flatten_df(df)

    print("\nFlattened Schema:")
    flat_df.printSchema()

    print("\nFlattened Data:")
    flat_df.show(truncate=False)



In [0]:
from pyspark.sql.types import StructType,StructField,MapType,StringType,LongType
schema= StructType([
    StructField("id", LongType(), True),
    StructField("name", MapType(StringType(), StringType()), True),
    StructField("contacts", ArrayType(
        StructType([
            StructField("type", StringType(), True),
            StructField("value", StringType(), True)
        ])
    ), True),
    StructField("address", MapType(StringType(), StringType()), True)
])
